# 📘 Notebook 10: Project: An ATM as a State Machine

### Course: *From Zero to Data Structures & Algorithms in Python*
**Instructor:** Ioannis Tsioulis

*Module B: Mini Projects · Notebook 3 of 3 in this module · (10 of 18 overall)*

---

## 🎯 Learning objectives

By the end of this notebook you will be able to:

- Explain what a **state machine** is: states, events and transitions
- Draw the behaviour of a program as a table of states before coding it
- Implement a state machine with one variable and a `while` loop
- Recognise why this structure scales better than deeply nested `if` statements
- Add new states and new rules to an existing machine
- Rewrite the machine as a **class** that owns its own state

> **Prerequisites:** Module A, especially Notebooks 02, 03, 06 and 07.
>
> 🔭 **Why this project matters:** state machines are everywhere: traffic lights, vending machines, lifts, network protocols, the screens of a mobile app, the enemies in a video game. Once you learn to see them, you will find that a surprising number of programs are best described this way.

## 1. What is a state machine?

### Intuition first
Think about how you use a cash machine. At every moment the machine is in exactly **one situation**: waiting for a card, waiting for a PIN, showing the menu, handing out cash. What it does next depends on two things only: **which situation it is in**, and **what you do**.

Typing `1234` means something completely different when the machine is asking for a PIN than when it is asking for an amount. The same input, a different situation, a different result.

### Formal definition
A **state machine** consists of:

- a finite set of **states** (the situations),
- **events** (inputs) that can occur in each state,
- **transitions**: rules of the form *"in state A, when event X happens, do something and move to state B"*,
- one **current state** at any moment.

### Why it matters
In Notebook 06 we wrote a small banking program with a single menu. A real ATM has a sequence of steps, some of which may fail and send the user back. Written with ordinary nested `if` statements, that logic quickly becomes a tangle. A state machine keeps it flat and readable: **one block of code per state**.

## 2. Designing the machine on paper

As in the previous project, we design first. Our ATM has six states.

| State | What the machine does | Possible next states |
|---|---|---|
| `INSERT_CARD` | asks for the card | `ENTER_PIN`, or stop |
| `ENTER_PIN` | asks for the PIN | `SELECT_ACTION` if correct, `INSERT_CARD` if wrong |
| `SELECT_ACTION` | shows the menu | `WITHDRAW`, `BALANCE`, `EJECT_CARD` |
| `WITHDRAW` | asks for an amount, checks the balance | `EJECT_CARD`, or back to `SELECT_ACTION` on bad input |
| `BALANCE` | shows the balance | `EJECT_CARD` |
| `EJECT_CARD` | returns the card | `INSERT_CARD`, or stop |

Read the table as a set of sentences: *"In state `ENTER_PIN`, if the PIN is correct, move to `SELECT_ACTION`."* Every row will become one branch of the program.

### The implementation idea
The whole machine needs just **one variable**, `current_state`, and one loop:

```python
current_state = "INSERT_CARD"

while True:
    if current_state == "INSERT_CARD":
        ...                          # handle this state, then choose the next one
        current_state = "ENTER_PIN"
    elif current_state == "ENTER_PIN":
        ...
```
On every pass, the loop looks at the current state, runs the code for that state, and updates `current_state`. Changing that variable **is** the transition.

## 3. Stage 1: a two-state machine

We start with the smallest possible machine, to see the mechanism clearly. It has two states, and the user can move between them.

In [ ]:
current_state = "INSERT_CARD"

while True:
    if current_state == "INSERT_CARD":
        print("Please insert your card.")
        action = input("Type 'insert' or 'exit': ")

        if action.lower() == "insert":
            current_state = "CARD_INSERTED"      # transition
        elif action.lower() == "exit":
            print("Goodbye!")
            break
        else:
            print("Invalid option!")             # no transition: same state again

    elif current_state == "CARD_INSERTED":
        print("Card accepted.")
        input("Press Enter to take your card back: ")
        current_state = "INSERT_CARD"            # transition

Three things can happen at the end of a branch, and you have just seen all of them:

1. `current_state` is changed: a **transition** to another state.
2. `current_state` is left alone: the machine **stays** in the same state and asks again.
3. `break`: the machine **stops**.

## 4. Stage 2: the complete ATM

Now we write one branch for each row of the design table. The program is longer, but it is not more complicated: every branch has the same simple shape as in Stage 1.

In [ ]:
current_state = "INSERT_CARD"
balance = 1000          # starting balance

while True:
    if current_state == "INSERT_CARD":
        print("\nPlease insert your card.")
        action = input("Type 'insert' or 'exit': ")

        if action.lower() == "insert":
            current_state = "ENTER_PIN"
        elif action.lower() == "exit":
            print("Goodbye!")
            break
        else:
            print("Invalid option!")

    elif current_state == "ENTER_PIN":
        pin = input("Enter your PIN: ")
        if pin == "1234":
            current_state = "SELECT_ACTION"
        else:
            print("Wrong PIN! Please try again.")
            current_state = "INSERT_CARD"

    elif current_state == "SELECT_ACTION":
        print("\nSelect an action:")
        print("1. Withdraw")
        print("2. Balance")
        print("3. Exit")
        choice = input("Choice: ")

        if choice == "1":
            current_state = "WITHDRAW"
        elif choice == "2":
            current_state = "BALANCE"
        elif choice == "3":
            current_state = "EJECT_CARD"
        else:
            print("Invalid option!")

    elif current_state == "WITHDRAW":
        amount = input("How much would you like to withdraw? ")
        if not amount.isdigit():
            print("Please enter a valid amount.")
            current_state = "SELECT_ACTION"
            continue

        amount = int(amount)
        if amount > balance:
            print("Insufficient funds!")
        else:
            balance -= amount
            print(f"Withdrawal of {amount} euros successful. Balance: {balance} euros")

        current_state = "EJECT_CARD"

    elif current_state == "BALANCE":
        print(f"Your balance is {balance} euros.")
        current_state = "EJECT_CARD"

    elif current_state == "EJECT_CARD":
        print("Please take your card.")
        again = input("Would you like another transaction? (y/n): ")
        if again.lower() == "y":
            current_state = "INSERT_CARD"
        else:
            print("Thank you! Have a nice day.")
            break

### Reading the code against the design
Compare the program with the table in section 2. Each row became one `elif` branch, and each entry in the last column became one assignment to `current_state`. Nothing else is needed.

Two details are worth pointing out.

- In `WITHDRAW`, the test `amount.isdigit()` validates the input **before** converting it, so the program survives an answer like `abc`. After a bad amount we use `continue` to jump straight to the next pass of the loop, in the state `SELECT_ACTION`.
- The `balance` variable is **not** a state. States describe *where we are in the conversation*. The balance is **data** that the machine carries along.

> ⚠️ **Common pitfalls**
>
> - A spelling mistake in a state name, such as `"ENTER_PN"`, creates a state that no branch handles. The loop then runs forever doing nothing. A final `else` branch that prints an error and stops is a useful safety net.
> - Forgetting to set `current_state` at the end of a branch keeps the machine in the same state, sometimes intentionally, often not.

## 5. Stage 3: extending the machine

The real test of a design is how easily it can be changed. Let us add a rule every real ATM has: **after three wrong PINs, the card is retained**.

This requires one new piece of data (the number of failed attempts) and one new state (`CARD_BLOCKED`). No existing branch other than `ENTER_PIN` needs to change. The cell below shows only the affected part of the machine, reduced to the PIN logic, so that the change is easy to see.

In [ ]:
current_state = "ENTER_PIN"
failed_attempts = 0

while True:
    if current_state == "ENTER_PIN":
        pin = input("Enter your PIN: ")
        if pin == "1234":
            failed_attempts = 0
            current_state = "SELECT_ACTION"
        else:
            failed_attempts += 1
            print(f"Wrong PIN! ({failed_attempts} of 3 attempts used)")
            if failed_attempts == 3:
                current_state = "CARD_BLOCKED"

    elif current_state == "CARD_BLOCKED":
        print("Too many wrong attempts. Your card has been retained.")
        break

    elif current_state == "SELECT_ACTION":
        print("PIN accepted. (The rest of the machine continues from here.)")
        break

## 6. Stage 4: the machine as a class

In Notebook 07 we said that a class bundles **data** with the **functions that work on it**. Our ATM has exactly that: data (`balance`, `current_state`) and behaviour (one piece of code per state).

In the version below, each state becomes a **method**, and a dictionary maps each state name to its method. The main loop shrinks to two lines: find the method for the current state, and call it.

In [ ]:
class ATM:
    def __init__(self, balance, pin):
        self.balance = balance
        self.pin = pin
        self.state = "ENTER_PIN"
        self.running = True

    def enter_pin(self):
        if input("Enter your PIN: ") == self.pin:
            self.state = "SELECT_ACTION"
        else:
            print("Wrong PIN!")

    def select_action(self):
        choice = input("1. Withdraw  2. Balance  3. Exit: ")
        if choice == "1":
            self.state = "WITHDRAW"
        elif choice == "2":
            print(f"Your balance is {self.balance} euros.")
        elif choice == "3":
            print("Please take your card. Goodbye!")
            self.running = False
        else:
            print("Invalid option!")

    def withdraw(self):
        amount = input("Amount: ")
        if not amount.isdigit():
            print("Please enter a valid amount.")
        elif int(amount) > self.balance:
            print("Insufficient funds!")
        else:
            self.balance -= int(amount)
            print(f"Please take your cash. Balance: {self.balance} euros")
        self.state = "SELECT_ACTION"

    def run(self):
        handlers = {
            "ENTER_PIN": self.enter_pin,
            "SELECT_ACTION": self.select_action,
            "WITHDRAW": self.withdraw,
        }
        while self.running:
            handlers[self.state]()        # look up the method for this state, and call it

atm = ATM(1000, "1234")
atm.run()

> 🧠 The line `handlers[self.state]()` deserves a second look. A dictionary can hold **functions** as values. We look one up by the name of the state and call it with `()`. This replaces the long `if` / `elif` chain entirely, and adding a state now means adding one method and one dictionary entry.

---
## ✏️ Exercises

The first exercises build new, small state machines. The later ones extend the ATM.

### Exercise 1 (Traffic light)
Model a traffic light with the states `GREEN`, `YELLOW` and `RED`, which follow each other in that order and then start again. Print the state 7 times, starting from `GREEN`. There is no user input in this machine.

In [ ]:
# Your solution here:


<details><summary>💡 Show solution</summary>

```python
current_state = "GREEN"

for step in range(7):
    print(current_state)
    if current_state == "GREEN":
        current_state = "YELLOW"
    elif current_state == "YELLOW":
        current_state = "RED"
    elif current_state == "RED":
        current_state = "GREEN"
```
</details>

### Exercise 2 (Traffic light with a dictionary)
Rewrite the traffic light so that the transitions are stored in a dictionary that maps each state to the next one. The loop body should need only two lines.

In [ ]:
# Your solution here:


<details><summary>💡 Show solution</summary>

```python
next_state = {"GREEN": "YELLOW", "YELLOW": "RED", "RED": "GREEN"}

current_state = "GREEN"
for step in range(7):
    print(current_state)
    current_state = next_state[current_state]
```

*When transitions do not depend on any input, a dictionary describes the whole machine. The transition table has literally become data.*
</details>

### Exercise 3 (A door)
A door can be `OPEN`, `CLOSED` or `LOCKED`. The commands are `open`, `close`, `lock` and `unlock`. Only sensible moves are allowed: you can lock only a closed door, you can open only a closed (not locked) door, and so on. Write a loop that reads commands until the user types `quit`, printing the new state, or `Not possible` for a command that makes no sense in the current state.

In [ ]:
# Your solution here:


<details><summary>💡 Show solution</summary>

```python
current_state = "CLOSED"

while True:
    command = input(f"The door is {current_state}. Command: ")
    if command == "quit":
        break

    if current_state == "CLOSED" and command == "open":
        current_state = "OPEN"
    elif current_state == "CLOSED" and command == "lock":
        current_state = "LOCKED"
    elif current_state == "OPEN" and command == "close":
        current_state = "CLOSED"
    elif current_state == "LOCKED" and command == "unlock":
        current_state = "CLOSED"
    else:
        print("Not possible")

print("Final state:", current_state)
```
</details>

### Exercise 4 (A deposit state)
Add a `DEPOSIT` option to the class-based `ATM` of section 6. Create a class `ATMWithDeposit` that **inherits** from `ATM` (Notebook 07, section 8) and define only what is new or different. It needs a `deposit` method, a new version of `select_action` with a fourth option, and a new `run` that includes the new state.

In [ ]:
# Your solution here:


<details><summary>💡 Show solution</summary>

```python
class ATMWithDeposit(ATM):
    def select_action(self):
        choice = input("1. Withdraw  2. Balance  3. Exit  4. Deposit: ")
        if choice == "4":
            self.state = "DEPOSIT"
        elif choice == "1":
            self.state = "WITHDRAW"
        elif choice == "2":
            print(f"Your balance is {self.balance} euros.")
        elif choice == "3":
            print("Please take your card. Goodbye!")
            self.running = False
        else:
            print("Invalid option!")

    def deposit(self):
        amount = input("Amount to deposit: ")
        if amount.isdigit() and int(amount) > 0:
            self.balance += int(amount)
            print(f"Deposited. Balance: {self.balance} euros")
        else:
            print("Please enter a valid amount.")
        self.state = "SELECT_ACTION"

    def run(self):
        handlers = {
            "ENTER_PIN": self.enter_pin,
            "SELECT_ACTION": self.select_action,
            "WITHDRAW": self.withdraw,
            "DEPOSIT": self.deposit,
        }
        while self.running:
            handlers[self.state]()

atm = ATMWithDeposit(1000, "1234")
atm.run()
```

*`ATMWithDeposit` receives `__init__`, `enter_pin` and `withdraw` unchanged from `ATM`, and overrides only `select_action` and `run`. Inheritance kept the change small.*
</details>

### Exercise 5 (A vending machine, a little harder)
Model a vending machine that sells one drink for 2 euros. States: `WAITING` (no money yet), `HAS_MONEY` (some money inserted) and `DISPENSING`. The user types `1` or `2` to insert a coin of that value, or `cancel` to get the money back. When the inserted total reaches 2 euros or more, the machine dispenses the drink, gives change if needed, and returns to `WAITING`. Stop when the user types `quit`.

In [ ]:
# Your solution here:


<details><summary>💡 Show solution</summary>

```python
current_state = "WAITING"
inserted = 0
PRICE = 2

while True:
    if current_state == "WAITING":
        command = input("Insert a coin (1 or 2), or 'quit': ")
        if command == "quit":
            break
        elif command in ["1", "2"]:
            inserted += int(command)
            current_state = "HAS_MONEY"
        else:
            print("Nothing to cancel.")

    elif current_state == "HAS_MONEY":
        if inserted >= PRICE:
            current_state = "DISPENSING"
            continue
        command = input(f"Inserted {inserted} euro. Insert a coin (1 or 2), or 'cancel': ")
        if command == "cancel":
            print(f"Returning {inserted} euro.")
            inserted = 0
            current_state = "WAITING"
        elif command in ["1", "2"]:
            inserted += int(command)

    elif current_state == "DISPENSING":
        print("Enjoy your drink!")
        if inserted > PRICE:
            print(f"Your change: {inserted - PRICE} euro.")
        inserted = 0
        current_state = "WAITING"
```

*`DISPENSING` is a state that needs no input: the machine passes through it and moves on. `inserted`, like the balance of the ATM, is data and not a state.*
</details>

## 🔑 Key takeaways

- A **state machine** has a set of **states**, exactly one **current state**, and **transitions** triggered by events.
- Design it on paper as a table: state, what happens there, and where it can go next.
- The implementation needs one variable and one loop: **one branch per state**, and changing the variable is the transition.
- The same input can mean different things in different states. That is the whole point.
- **States** describe where we are in the process; other values, like a balance, are **data**.
- A well-structured machine is easy to **extend**: a new state is a new branch, and the others stay untouched.
- As a class, each state becomes a method, and a dictionary of methods replaces the `if` chain.

### 🏁 You have completed Module B: Mini Projects!

You have built two games, an application that saves its data, and a state machine. More importantly, you have practised the method: design first, build in stages, give each function one job, test every piece.

So far we have asked only one question of our programs: *does it work?* From here on we ask a second one: ***how fast is it?***

**Next:** *Module C: Algorithms & Complexity*, beginning with **Big O Notation** (Notebook 11), where we take out a stopwatch and measure.

---
*End of Module B.*

---
*© Ioannis Tsioulis. *From Zero to Data Structures & Algorithms in Python*. Prepared for educational use.*